## MAS (Multi Agent System)

MAS를 이용해...

In [2]:
import os
from dotenv import load_dotenv

from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


# .env 파일로부터 api_key를 불러오는 코드
load_dotenv(dotenv_path="../.env")

# 모델 이름
model_name = "gpt-5.4-mini"

# LLM 정의
model = init_chat_model(
    model=model_name,
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.7,
)

In [3]:
weather_agent = create_agent(
    model,
    system_prompt="당신은 여행지 날씨 전문가입니다. 도시명을 받으면 간단히 날씨와 적합한 옷차림을 알려주세요.",
    tools=[],
)

spots_agent = create_agent(
    model,
    system_prompt="당신은 여행지 관광명소 전문가입니다. 도시명을 받으면 TOP 3 명소를 추천해주세요.",
    tools=[],
)

food_agent = create_agent(
    model,
    system_prompt="당신은 여행지 음식 전문가입니다. 도시명을 받으면 꼭 먹어야 할 음식 3가지를 알려주세요.",
    tools=[],
)

In [8]:
from langchain.tools import tool


@tool
def ask_weather(city: str) -> str:
    """
    여행지의 날씨와 옷차림을 알려줍니다.
    """
    result = weather_agent.invoke({
        "messages": [{"role": "user", "content": city}]
    })
    return result["messages"][-1].content


@tool
def ask_spots(city: str) -> str:
    """
    여행지의 관광명소 TOP 3를 추천합니다.
    """
    result = spots_agent.invoke({
        "messages": [{"role": "user", "content": city}]
    })
    return result["messages"][-1].content


@tool
def ask_food(city: str) -> str:
    """
    여행지에서 꼭 먹어야 할 음식을 추천합니다.
    """
    result = food_agent.invoke({
        "messages": [{"role": "user", "content": city}]
    })
    return result["messages"][-1].content

In [ ]:
from langchain.agents.middleware import wrap_tool_call


@wrap_tool_call
def logging_middleware(request, handler): # request와 handler 2개의 인자를 반드시 입력 받아야 합니다!
    # request : 도구 호출 정보
    # handler : 실제 도구 실행 함수
    tool_name = request.tool_call["name"] # tool 이름
    tool_args = request.tool_call["args"] # tool의 입력 인자
    print(f"[호출] 도구: {tool_name} | 인수: {tool_args}")
    try:
        result = handler(request)
        print(f"[성공]")
        return result
    except Exception as e:
        print(f"[실패] 오류: {e}")
        raise

In [17]:
main_agent = create_agent(
    model,
    tools=[ask_weather, ask_spots, ask_food],   # subagent를 도구로 등록
    middleware=[logging_middleware],
    system_prompt=(
        "당신은 여행 계획 전문가입니다. "
        "사용자의 여행 요청을 받으면 날씨, 명소, 음식 전문가에게 물어본 뒤 "
        "종합해서 멋진 여행 계획을 만들어 주세요."
    ),
)

In [18]:
response = main_agent.invoke({
    "messages": [{"role": "user", "content": "도쿄 3박 4일 여행 계획 짜줘"}]
})

[호출] 도구: ask_weather | 인수: {'city': '도쿄'}
[호출] 도구: ask_spots | 인수: {'city': '도쿄'}
[호출] 도구: ask_food | 인수: {'city': '도쿄'}
[성공] 결과: 도쿄에서 꼭 가볼 만한 TOP 3 명소를 추천드릴게요.

1. **센소지(浅草寺)**
   - 도쿄에서 가장 오래된 사찰 중 하나로, 아사쿠사 지역의 상징입니다.
   - 나카미세 거리에서 간식, 기념품 구경도 함께 즐기기 좋아요.

2. **도쿄 스카이트리**
   - 도쿄 전경을 한눈에 볼 수 있는 대표 전망대입니다.
   - 야경이 특히 아름답고, 쇼핑·식사까지 한 번에 즐길 수 있습니다.

3. **시부야 스크램블 교차로 & 시부야 지역**
   - 세계적으로 유명한 번화가 풍경을 직접 볼 수 있는 곳입니다.
   - 쇼핑, 카페, 젊은 분위기, 하치코 동상까지 함께 둘러보기 좋아요.

원하시면 제가 **“도쿄 TOP 3를 여행 테마별(쇼핑/야경/전통)”**로 다시 추천해드릴게요.
[성공] 결과: 도쿄에서 꼭 먹어야 할 음식 3가지입니다.

1. **스시**
   - 도쿄는 신선한 해산물과 정교한 스시 문화로 유명합니다.
   - 츠키지, 스시 전문점, 카운터 스시에서 꼭 한 번 맛보세요.

2. **라멘**
   - 도쿄식 간장 라멘은 깔끔하고 깊은 맛이 특징입니다.
   - 전통적인 쇼유 라멘부터 현대적인 인기 라멘집까지 다양하게 즐길 수 있어요.

3. **텐동**
   - 바삭하게 튀긴 새우, 채소, 생선을 밥 위에 올리고 달콤짭짤한 소스를 뿌린 덮밥입니다.
   - 부담 없이 도쿄의 일상적인 맛을 느끼기 좋습니다.

원하시면 **“도쿄에서 이 음식들을 잘하는 추천 맛집”**도 알려드릴게요.
[성공] 결과: 도쿄는 보통 **해양성 기후**라서 계절에 따라 꽤 달라요.

- **봄/가을:** 선선하고 쾌적함  
  → **긴팔 + 얇은 겉옷** 추천
- **여름:** 덥고 습함, 비도 잦음  
  → **반팔, 통풍 잘

In [19]:
print(response['messages'][-1].content)

좋아요! 도쿄 **3박 4일 여행 계획**을 날씨·명소·음식까지 반영해서 짜드릴게요.  
도쿄는 계절에 따라 옷차림이 꽤 달라서, 아래는 **기본형 일정 + 계절 공통 팁**으로 구성했어요.

## 여행 전 한줄 요약
- **도쿄 날씨/옷차림:**  
  - 봄/가을: 긴팔 + 얇은 겉옷  
  - 여름: 반팔, 통풍 잘 되는 옷, 우산  
  - 겨울: 코트, 니트, 목도리
- **핵심 명소:** 센소지, 도쿄 스카이트리, 시부야 스크램블 교차로
- **꼭 먹을 음식:** 스시, 라멘, 텐동

---

# 도쿄 3박 4일 일정

## 1일차: 도착 + 아사쿠사 전통 분위기
**테마:** 도쿄 첫인상, 전통 산책

### 오전/이동
- 공항 도착 후 숙소 체크인
- 시간이 여유 있으면 주변 편의점/카페에서 간단히 요기

### 오후
- **센소지(아사쿠사)** 방문  
  - 도쿄에서 가장 오래된 사찰 중 하나
  - 나카미세 거리 구경하면서 일본식 간식, 기념품 쇼핑

### 저녁
- 아사쿠사 근처에서 **첫 식사: 텐동** 추천
- 밤에는 **스카이트리 야경**까지 보면 일정이 아주 좋습니다

### 추천 포인트
- 첫날은 너무 빡빡하게 잡지 말고, 걷기 중심으로 가볍게 시작하기 좋아요.

---

## 2일차: 도쿄 대표 랜드마크 + 야경
**테마:** 전망, 쇼핑, 도쿄다운 풍경

### 오전
- **도쿄 스카이트리**  
  - 전망대에서 도쿄 전경 감상
  - 날씨가 맑으면 멀리까지 잘 보여서 만족도가 높아요

### 점심
- 스카이트리 주변에서 가볍게 식사
- 또는 근처에서 **라멘** 한 끼 추천

### 오후
- 우에노나 긴자 쪽으로 이동해 쇼핑/산책
- 시간이 있으면 카페 투어 또는 백화점 구경

### 저녁
- 도쿄식 **스시** 즐기기
- 가능하면 카운터 스시나 회전초밥도 좋습니다

### 밤
- 숙소 복귀 전 시내 야경 산책

---

## 3일차: 시부야 + 젊은 감성 + 쇼핑
**테마:** 활기찬 도쿄, 사진 명소

### 오전
- 